# B7 · Du notebook à la production — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b7_production/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/b7_production/03_notebook.ipynb`, créée par `python tools/start_chapter.py B7`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B7.Q1 à B7.Q11, B7.R1 à B7.R3, B7.1 à B7.4 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | B7.9 à B7.15 | le projet : du notebook au paquet, ligne de commande, configuration, reproductibilité, suivi des runs et tests (*prochaine session de génération*) | | | |
| B | B7.16 à B7.21 | le service : les écarts entre entraînement et production, l'artefact, l'API, la latence, l'accélération sur CPU et l'export (*prochaine session de génération*) | | | |
| C | B7.22 à B7.25 | l'exploitation : la dérive, l'intégration continue, le Dockerfile et le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B7")  # your own library

## Objectifs et rappel express

- Restructurer un notebook en paquet Python installable, avec une ligne de commande, du logging et une configuration validée.
- Rendre un entraînement reproductible et traçable (graines, déterminisme, versions, empreinte des données, journal des runs), et rapporter un score sur plusieurs graines avec son intervalle.
- Écrire des tests utiles à un projet de ML et les automatiser dans une intégration continue.
- Empaqueter un modèle avec son prétraitement, le servir par une API HTTP et mesurer sa latence.
- Exporter et accélérer l'inférence sur CPU en vérifiant l'équivalence numérique ; détecter une dérive des données et décider d'une action.

**Rappel express.** Intervalle sur $n$ graines : $\bar a \pm t_{0{,}975;\,n-1}\,\frac{s}{\sqrt n}$ ($s$ divisé par $n - 1$). Percentile de `np.percentile` : position $h = \frac{q}{100}(n - 1)$ dans les mesures triées, interpolation linéaire. Débit $= \frac{\text{taille du batch}}{\text{latence du batch}}$ ; loi de Little $L = \lambda W$. Taille des poids $= N \times$ octets par paramètre (4, 2, 1). $\operatorname{PSI} = \sum_k (p_k - q_k)\ln\frac{p_k}{q_k} = D_{\text{KL}}(p\,\|\,q) + D_{\text{KL}}(q\,\|\,p)$ ; $D = \sup_x |F_{\text{réf}}(x) - F_{\text{prod}}(x)|$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B7.1 à B7.4)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; plusieurs choix : les lettres ensemble, entre guillemets, dans n'importe quel ordre (`"EF"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », l'estimation 🧮 B7.5, la réflexion (🗣️ B7.6, ⚖️ B7.7, 📄 B7.8) et l'entretien se corrigent avec `05_solutions.md`.

**B7.Q1 — Les pièges du notebook : état caché et ordre d'exécution**

In [ ]:
# B7.Q1: the values you found on paper
answer_B7_Q1a = ...  # a) a whole number
answer_B7_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q1c = ...  # c) True or False
answer_B7_Q1d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B7_Q1a, answer_B7_Q1b, answer_B7_Q1c, answer_B7_Q1d, answer_B7_Q1e]):
    wb.check(f"B7.Q1{letter}", answer)

**B7.Q2 — src/, tests/, configs/, scripts/ : où va quoi ?**

In [ ]:
# B7.Q2: the values you found on paper
answer_B7_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q2c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_Q2e = ...  # e) True or False
answer_B7_Q2f = ...  # f) the letter of your choice, e.g. "E"
answer_B7_Q2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B7_Q2a, answer_B7_Q2b, answer_B7_Q2c, answer_B7_Q2d, answer_B7_Q2e, answer_B7_Q2f, answer_B7_Q2g]):
    wb.check(f"B7.Q2{letter}", answer)

**B7.Q3 — print ou logging : niveaux, destinations et messages utiles**

In [ ]:
# B7.Q3: the values you found on paper
answer_B7_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q3b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q3c = ...  # c) a whole number
answer_B7_Q3d = ...  # d) True or False
answer_B7_Q3e = ...  # e) the letter of your choice, e.g. "E"
answer_B7_Q3f = ...  # f) a whole number

for letter, answer in zip("abcdef", [answer_B7_Q3a, answer_B7_Q3b, answer_B7_Q3c, answer_B7_Q3d, answer_B7_Q3e, answer_B7_Q3f]):
    wb.check(f"B7.Q3{letter}", answer)

**B7.Q4 — Ce qui va dans la configuration, et ce qui ne doit jamais y aller**

In [ ]:
# B7.Q4: the values you found on paper
answer_B7_Q4a = ...  # a) the letters, e.g. "EF"
answer_B7_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q4c = ...  # c) a whole number
answer_B7_Q4d = ...  # d) True or False
answer_B7_Q4e = ...  # e) the letter of your choice, e.g. "E"
answer_B7_Q4f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B7_Q4a, answer_B7_Q4b, answer_B7_Q4c, answer_B7_Q4d, answer_B7_Q4e, answer_B7_Q4f]):
    wb.check(f"B7.Q4{letter}", answer)

**B7.Q5 — Même graine, résultats différents : les sources de non-déterminisme**

In [ ]:
# B7.Q5: the values you found on paper
answer_B7_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q5b = ...  # b) True or False
answer_B7_Q5c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_Q5d = ...  # d) True or False
answer_B7_Q5e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B7_Q5a, answer_B7_Q5b, answer_B7_Q5c, answer_B7_Q5d, answer_B7_Q5e]):
    wb.check(f"B7.Q5{letter}", answer)

**B7.Q6 — requirements.txt, fichier de verrouillage, pyproject.toml : qui fige quoi ?**

In [ ]:
# B7.Q6: the values you found on paper
answer_B7_Q6a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q6b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q6c = ...  # c) True or False
answer_B7_Q6d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B7_Q6a, answer_B7_Q6b, answer_B7_Q6c, answer_B7_Q6d, answer_B7_Q6e]):
    wb.check(f"B7.Q6{letter}", answer)

**B7.Q7 — Que tester dans un projet de machine learning ?**

In [ ]:
# B7.Q7: the values you found on paper
answer_B7_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q7b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q7c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_Q7d = ...  # d) True or False
answer_B7_Q7e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B7_Q7a, answer_B7_Q7b, answer_B7_Q7c, answer_B7_Q7d, answer_B7_Q7e]):
    wb.check(f"B7.Q7{letter}", answer)

**B7.Q8 — Artefact de modèle : les poids seuls ne suffisent pas**

In [ ]:
# B7.Q8: the values you found on paper
answer_B7_Q8a = ...  # a) the letters, e.g. "EF"
answer_B7_Q8b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q8c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_Q8d = ...  # d) True or False
answer_B7_Q8e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B7_Q8a, answer_B7_Q8b, answer_B7_Q8c, answer_B7_Q8d, answer_B7_Q8e]):
    wb.check(f"B7.Q8{letter}", answer)

**B7.Q9 — Latence p50, p95 et débit : ce que ressent l'utilisateur**

In [ ]:
# B7.Q9: the values you found on paper
answer_B7_Q9a = ...  # a) a whole number
answer_B7_Q9b = ...  # b) a whole number
answer_B7_Q9c = ...  # c) True or False
answer_B7_Q9d = ...  # d) a whole number
answer_B7_Q9e = ...  # e) a whole number
answer_B7_Q9f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_B7_Q9a, answer_B7_Q9b, answer_B7_Q9c, answer_B7_Q9d, answer_B7_Q9e, answer_B7_Q9f]):
    wb.check(f"B7.Q9{letter}", answer)

**B7.Q10 — torch.export, ONNX, TorchScript : lequel choisir aujourd'hui ?**

In [ ]:
# B7.Q10: the values you found on paper
answer_B7_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_Q10c = ...  # c) True or False
answer_B7_Q10d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_Q10e = ...  # e) True or False
answer_B7_Q10f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_B7_Q10a, answer_B7_Q10b, answer_B7_Q10c, answer_B7_Q10d, answer_B7_Q10e, answer_B7_Q10f]):
    wb.check(f"B7.Q10{letter}", answer)

**B7.Q11 — Écart entraînement-production et dérive des données : vrai ou faux**

In [ ]:
# B7.Q11: the values you found on paper
answer_B7_Q11a = ...  # a) True or False
answer_B7_Q11b = ...  # b) True or False
answer_B7_Q11c = ...  # c) True or False
answer_B7_Q11d = ...  # d) True or False
answer_B7_Q11e = ...  # e) the letter of your choice, e.g. "E"
answer_B7_Q11f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_B7_Q11a, answer_B7_Q11b, answer_B7_Q11c, answer_B7_Q11d, answer_B7_Q11e, answer_B7_Q11f]):
    wb.check(f"B7.Q11{letter}", answer)

**B7.R1 — Ce qu'une ligne de journal d'expériences doit contenir**

In [ ]:
# B7.R1: the values you found on paper
answer_B7_R1a = ...  # a) the letters, e.g. "EF"
answer_B7_R1b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_R1c = ...  # c) True or False
answer_B7_R1d = ...  # d) a whole number
answer_B7_R1e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B7_R1a, answer_B7_R1b, answer_B7_R1c, answer_B7_R1d, answer_B7_R1e]):
    wb.check(f"B7.R1{letter}", answer)

**B7.R2 — Checkpoint complet, state_dict et weights_only**

In [ ]:
# B7.R2: the values you found on paper
answer_B7_R2a = ...  # a) the letters, e.g. "EF"
answer_B7_R2b = ...  # b) the letter of your choice, e.g. "E"
answer_B7_R2c = ...  # c) True or False
answer_B7_R2d = ...  # d) True or False
answer_B7_R2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_B7_R2a, answer_B7_R2b, answer_B7_R2c, answer_B7_R2d, answer_B7_R2e]):
    wb.check(f"B7.R2{letter}", answer)

**B7.R3 — Sauvegarder un Pipeline et figer les versions**

In [ ]:
# B7.R3: the values you found on paper
answer_B7_R3a = ...  # a) the letter of your choice, e.g. "E"
answer_B7_R3b = ...  # b) True or False
answer_B7_R3c = ...  # c) the letter of your choice, e.g. "E"
answer_B7_R3d = ...  # d) True or False
answer_B7_R3e = ...  # e) the letters, e.g. "EF"
answer_B7_R3f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_B7_R3a, answer_B7_R3b, answer_B7_R3c, answer_B7_R3d, answer_B7_R3e, answer_B7_R3f]):
    wb.check(f"B7.R3{letter}", answer)

**Ex B7.1 — Taille d'un artefact : float32, bfloat16, int8, et la mémoire au chargement**

In [ ]:
# Ex B7.1: the values you found on paper
answer_B7_1a = ...  # a) a number, 1 decimal
answer_B7_1b = ...  # b) a number, 1 decimal
answer_B7_1c = ...  # c) a number, 1 decimal
answer_B7_1d = ...  # d) a number, 1 decimal
answer_B7_1e = ...  # e) a number, 1 decimal
answer_B7_1f = ...  # f) a number, 1 decimal
answer_B7_1g = ...  # g) a number, 1 decimal
answer_B7_1h = ...  # h) a whole number
answer_B7_1i = ...  # i) a whole number
answer_B7_1j = ...  # j) a number, 1 decimal

for letter, answer in zip("abcdefghij", [answer_B7_1a, answer_B7_1b, answer_B7_1c, answer_B7_1d, answer_B7_1e, answer_B7_1f, answer_B7_1g, answer_B7_1h, answer_B7_1i, answer_B7_1j]):
    wb.check(f"B7.1{letter}", answer)

**Ex B7.2 — Cinq graines, un intervalle : la dispersion d'une accuracy rapportée**

In [ ]:
# Ex B7.2: the values you found on paper
answer_B7_2a = ...  # a) a number, 1 decimal
answer_B7_2b = ...  # b) a number, 3 decimals
answer_B7_2c = ...  # c) a number, 3 decimals
answer_B7_2d = ...  # d) a number, 2 decimals
answer_B7_2e = ...  # e) a number, 2 decimals
answer_B7_2f = ...  # f) True or False
answer_B7_2g = ...  # g) a number, 1 decimal
answer_B7_2h = ...  # h) a number, 2 decimals
answer_B7_2i = ...  # i) True or False

for letter, answer in zip("abcdefghi", [answer_B7_2a, answer_B7_2b, answer_B7_2c, answer_B7_2d, answer_B7_2e, answer_B7_2f, answer_B7_2g, answer_B7_2h, answer_B7_2i]):
    wb.check(f"B7.2{letter}", answer)

**Ex B7.3 — Latence et débit : p50 et p95 sur vingt mesures, batch 1 contre batch 32**

In [ ]:
# Ex B7.3: the values you found on paper
answer_B7_3a = ...  # a) a number, 2 decimals
answer_B7_3b = ...  # b) a number, 2 decimals
answer_B7_3c = ...  # c) a number, 2 decimals
answer_B7_3d = ...  # d) a whole number
answer_B7_3e = ...  # e) a whole number
answer_B7_3f = ...  # f) a whole number
answer_B7_3g = ...  # g) a number, 1 decimal
answer_B7_3h = ...  # h) a number, 2 decimals
answer_B7_3i = ...  # i) True or False
answer_B7_3j = ...  # j) a number, 2 decimals

for letter, answer in zip("abcdefghij", [answer_B7_3a, answer_B7_3b, answer_B7_3c, answer_B7_3d, answer_B7_3e, answer_B7_3f, answer_B7_3g, answer_B7_3h, answer_B7_3i, answer_B7_3j]):
    wb.check(f"B7.3{letter}", answer)

**Ex B7.4 — Indice de stabilité de population (PSI) à la main sur quatre intervalles**

In [ ]:
# Ex B7.4: the values you found on paper
answer_B7_4a = ...  # a) a number, 4 decimals
answer_B7_4b = ...  # b) a number, 3 decimals
answer_B7_4c = ...  # c) a number, 3 decimals
answer_B7_4d = ...  # d) the letter of your choice, e.g. "E"
answer_B7_4e = ...  # e) a number, 3 decimals
answer_B7_4f = ...  # f) the letter of your choice, e.g. "E"
answer_B7_4g = ...  # g) True or False
answer_B7_4h = ...  # h) a number, 2 decimals
answer_B7_4i = ...  # i) a number, 3 decimals

for letter, answer in zip("abcdefghi", [answer_B7_4a, answer_B7_4b, answer_B7_4c, answer_B7_4d, answer_B7_4e, answer_B7_4f, answer_B7_4g, answer_B7_4h, answer_B7_4i]):
    wb.check(f"B7.4{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire où va chaque fichier d'un projet de ML (code, tests, configuration, données) et pourquoi un notebook ne suffit pas ?
2. Sais-tu rapporter une accuracy sur cinq graines avec son intervalle, et comparer deux modèles graine par graine ?
3. Sais-tu calculer un p50, un p95 et un débit, et dimensionner un service à la louche ?
4. Sais-tu calculer un PSI, dire ce qu'il mesure, et ce que tu ferais d'une alerte ?

**Pour aller plus loin** : l'article de Sculley et coll. (📄 B7.8) et le ML Test Score de Breck et coll., cités dans la fiche. La suite : le bonus B8, l'apprentissage par renforcement moderne.

*Les exercices de code B7.9 à B7.25 (le projet, le service, l'exploitation) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py B7` pour obtenir le notebook complet.*